In [ ]:


from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    recall_score, roc_auc_score, confusion_matrix
)
from xgboost import XGBClassifier

SEED = 13


N_OUTER_SPLITS = 5
N_OUTER_REPEATS = 5


N_INNER_SPLITS = 5
N_INNER_REPEATS = 4

BASE = Path(r"C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2")
PREPARED = BASE / "Prepared"
INPUT_CSV = PREPARED / "04_only_eeg.csv"

PARAM_GRID = {
    "xgb__max_depth": list(range(3, 12)),
    "xgb__learning_rate": [0.001, 0.01, 0.1, 1.0],
    "xgb__n_estimators": [50, 100],
}

OUT_DIR = PREPARED / "EEG_XGBoost_Conversation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FOLDS_CSV = OUT_DIR / "eeg_outer_folds_by_subject_repeated.csv"

In [ ]:

# CARGAR DATOS EEG


df = pd.read_csv(INPUT_CSV)
df["subject_id"] = df["subject_id"].astype(str).str.upper().str.strip()
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

# Features EEG: deben ser las 27 columnas numéricas de EEG.
meta_cols = {"subject_id", "avatar", "label", "phq", "outer_fold", "outer_repeat"}
feature_cols = [
    c for c in df.columns
    if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])
]


missing_total = int(df[feature_cols].isna().sum().sum())
print("Valores faltantes en features EEG:", missing_total)
if missing_total > 0:
    raise ValueError("Hay valores faltantes en EEG. Revisa la base de datos o activa imputación.")

print("Filas EEG:", len(df))
print("Sujetos EEG:", df["subject_id"].nunique())
print("Variables EEG:", len(feature_cols))
print("Clases por sujeto:")
display(df.drop_duplicates("subject_id")["label"].value_counts().sort_index())
print("Conversaciones por sujeto:")
display(df.groupby("subject_id").size().value_counts().sort_index())

Valores faltantes en features EEG: 0
Filas EEG: 558
Sujetos EEG: 94
Variables EEG: 27
Clases por sujeto:


label
0    55
1    39
Name: count, dtype: int64

Conversaciones por sujeto:


3     2
6    92
Name: count, dtype: int64

In [ ]:

# FUNCIONES 


def get_repeated_group_splits(X, y, groups, n_splits=N_INNER_SPLITS, n_repeats=N_INNER_REPEATS, seed=SEED):
    """Inner CV repetida, estratificada y agrupada por sujeto."""
    splits = []
    for rep in range(n_repeats):
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed + rep)
        splits.extend(list(cv.split(X, y, groups)))
    return splits


def make_outer_folds(subjects):
    """Outer CV: 5 folds repetido 5 veces, estratificado por sujeto."""
    rows = []
    for rep in range(1, N_OUTER_REPEATS + 1):
        cv = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=SEED + rep)
        for fold, (_, test_idx) in enumerate(cv.split(subjects["subject_id"], subjects["label"]), start=1):
            tmp = subjects.iloc[test_idx].copy()
            tmp["outer_repeat"] = rep
            tmp["outer_fold"] = fold
            rows.append(tmp)
    return pd.concat(rows, ignore_index=True)


def make_sample_weight(y):
    """Pesos inversos a la frecuencia de clase dentro del train fold."""
    counts = y.value_counts()
    return y.map({cls: len(y) / (len(counts) * n) for cls, n in counts.items()}).values


def make_model():
    """Pipeline: escalado dentro del fold + XGBoost."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("xgb", XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            random_state=SEED,
            n_jobs=-1,
        )),
    ])


def safe_auc(y_true, y_prob):
    try:
        return roc_auc_score(y_true, y_prob)
    except ValueError:
        return np.nan


def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": safe_auc(y_true, y_prob),
        "recall_pos": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_neg": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }


def summarize_metrics(metrics_df):
    cols = ["accuracy", "balanced_accuracy", "f1", "auc", "recall_pos", "recall_neg"]
    summary = pd.DataFrame({
        "mean": metrics_df[cols].mean(),
        "std": metrics_df[cols].std(),
    }).round(3)
    return summary

In [ ]:

# CREAR FOLDS EXTERNOS


subjects = df[["subject_id", "label"]].drop_duplicates("subject_id").reset_index(drop=True)
folds = make_outer_folds(subjects)
folds.to_csv(FOLDS_CSV, index=False)

print("Particiones externas:", folds[["outer_repeat", "outer_fold"]].drop_duplicates().shape[0])
display(pd.crosstab([folds["outer_repeat"], folds["outer_fold"]], folds["label"]).head(10))

Particiones externas: 25


label                     0  1
outer_repeat outer_fold       
1            1           11  8
             2           11  8
             3           11  8
             4           11  8
             5           11  7
2            1           11  8
             2           11  8
             3           11  8
             4           11  8
             5           11  7

In [ ]:

# NESTED CV CONVERSATION-LEVEL

all_predictions = []
all_metrics = []

for (rep, fold), test_subjects in folds.groupby(["outer_repeat", "outer_fold"]):
    print(f"\n===== REPEAT {rep} | FOLD {fold} =====")

    test_ids = set(test_subjects["subject_id"])
    train_df = df[~df["subject_id"].isin(test_ids)].copy()
    test_df = df[df["subject_id"].isin(test_ids)].copy()

    X_train = train_df[feature_cols]
    y_train = train_df["label"]
    groups_train = train_df["subject_id"]
    X_test = test_df[feature_cols]

    grid = GridSearchCV(
        estimator=make_model(),
        param_grid=PARAM_GRID,
        scoring="roc_auc",
        cv=get_repeated_group_splits(X_train, y_train, groups_train),
        n_jobs=-1,
        refit=True,
    )

    grid.fit(X_train, y_train, xgb__sample_weight=make_sample_weight(y_train))

    test_df = test_df.copy()
    test_df["y_prob"] = grid.best_estimator_.predict_proba(X_test)[:, 1]

    # Una predicción final por sujeto: media de probabilidades de sus conversaciones.
    pred_subject = test_df.groupby(["subject_id", "label"], as_index=False)["y_prob"].mean()
    pred_subject["y_pred"] = (pred_subject["y_prob"] >= 0.5).astype(int)
    pred_subject["outer_repeat"] = rep
    pred_subject["outer_fold"] = fold

    m = compute_metrics(pred_subject["label"], pred_subject["y_pred"], pred_subject["y_prob"])
    m.update({
        "outer_repeat": rep,
        "outer_fold": fold,
        "best_inner_auc": grid.best_score_,
        "best_max_depth": grid.best_params_["xgb__max_depth"],
        "best_learning_rate": grid.best_params_["xgb__learning_rate"],
        "best_n_estimators": grid.best_params_["xgb__n_estimators"],
    })

    all_predictions.append(pred_subject)
    all_metrics.append(m)

    print("Best AUC inner:", round(grid.best_score_, 3))
    print("Subject AUC:", round(m["auc"], 3))

print("\nProceso terminado.")


===== REPEAT 1 | FOLD 1 =====
Best AUC inner: 0.689
Subject AUC: 0.727

===== REPEAT 1 | FOLD 2 =====
Best AUC inner: 0.672
Subject AUC: 0.75

===== REPEAT 1 | FOLD 3 =====
Best AUC inner: 0.609
Subject AUC: 0.875

===== REPEAT 1 | FOLD 4 =====
Best AUC inner: 0.642
Subject AUC: 0.727

===== REPEAT 1 | FOLD 5 =====
Best AUC inner: 0.67
Subject AUC: 0.688

===== REPEAT 2 | FOLD 1 =====
Best AUC inner: 0.663
Subject AUC: 0.636

===== REPEAT 2 | FOLD 2 =====
Best AUC inner: 0.607
Subject AUC: 0.852

===== REPEAT 2 | FOLD 3 =====
Best AUC inner: 0.624
Subject AUC: 0.818

===== REPEAT 2 | FOLD 4 =====
Best AUC inner: 0.691
Subject AUC: 0.739

===== REPEAT 2 | FOLD 5 =====
Best AUC inner: 0.687
Subject AUC: 0.506

===== REPEAT 3 | FOLD 1 =====
Best AUC inner: 0.654
Subject AUC: 0.693

===== REPEAT 3 | FOLD 2 =====
Best AUC inner: 0.627
Subject AUC: 0.795

===== REPEAT 3 | FOLD 3 =====
Best AUC inner: 0.631
Subject AUC: 0.852

===== REPEAT 3 | FOLD 4 =====
Best AUC inner: 0.646
Subject AUC: 

In [ ]:

# GUARDAMOS RESULTADOS


predictions = pd.concat(all_predictions, ignore_index=True)
metrics_df = pd.DataFrame(all_metrics)

predictions.to_csv(OUT_DIR / "eeg_conversation_predictions_subject_level.csv", index=False)
metrics_df.to_csv(OUT_DIR / "eeg_conversation_outer_metrics.csv", index=False)

print("Archivos guardados en:", OUT_DIR)
print("\nMétricas principales: media ± desviación por outer fold")
display(summarize_metrics(metrics_df))

Archivos guardados en: C:\Users\oieru\OneDrive\Escritorio\MASTER\00-TFM\00-TFM_FINAL\DataBase\Limpio2\Prepared\EEG_XGBoost_Conversation_PaperLike_Repeat5_NoKNN

Métricas principales: media ± desviación por outer fold


,mean,std
accuracy,0.685,0.094
balanced_accuracy,0.666,0.105
f1,0.581,0.154
auc,0.716,0.109
recall_pos,0.554,0.194
recall_neg,0.778,0.098
